In [1]:
// load the formatting setup and the style sheet
#load "../setup.fsx"
HTML(System.IO.File.ReadAllText "../style.css")

The below script needs to be able to find the current output cell; this is an easy method to get it.

# Integer Square Root

In this notebook we work with very big numbers.  Therefore, we use the type `bigint`, which represents integers of arbitrary size.  Literals of type `bigint` are written with the suffix `I`, e.g. `0I`.  For integers, the operator `/` computes the integer division, i.e. it corresponds to the operator `//` in Python.

The function `isqrt` takes one argument:
- `n` is a natural number.

It returns the largest natural number $r$ such that
$r^2 \leq n$, i.e. we have
$$ \texttt{isqrt}(n) := \max\bigl(\{ r \in \mathbb{N} \mid r^2 \leq n \}\bigr). $$

**Implementation:** The result is computed recursively via a *divide-and-conquer* algorithm as follows:
1. $\texttt{isqrt}(0) = 0$.
2. $\bigl(2 \cdot \texttt{isqrt}(n \,\texttt{/}\, 4) + 1\bigr)^2 \leq n \rightarrow \texttt{isqrt}(n) = 2 \cdot \texttt{isqrt}(n \,\texttt{/}\, 4) + 1$.
3. $\bigl(2 \cdot \texttt{isqrt}(n \,\texttt{/}\, 4) + 1\bigr)^2 > n \rightarrow \texttt{isqrt}(n) = 2 \cdot \texttt{isqrt}(n \,\texttt{/}\, 4)$.

Here, $n \,\texttt{/}\, 4$ denotes the integer division of $n$ by $4$.

In [2]:
let rec isqrt (n: bigint) : bigint =
    if n = 0I then
        0I
    else
        let r = isqrt (n / 4I)
        if pown (2I * r + 1I) 2 <= n then
            2I * r + 1I
        else
            2I * r

In [3]:
for n in 0 .. 9 do
    printfn "isqrt(%d) = %A" n (isqrt (bigint n))

isqrt(

0

) = 

0

isqrt(

...

2

isqrt(

9

) = 

3

In order to test our implementation more thoroughly we use random numbers.  We create a random number generator with the seed `0`, so that the tests are reproducible.

In [4]:
let rnd = System.Random(0)

The function `runTests` takes one argument:
- `noTests` is the number of tests that are to be performed.

The function generates `noTests` random integers `n` such that $0 \leq n < 2^{64}$ and tests, whether 
`isqrt n` is the *integer square root* of `n` in each case.  If a test fails, an exception is raised.

**Implementation:** The method `rnd.NextInt64()` returns a random number $x$ with $0 \leq x < 2^{63}$.  To get a random number less than $2^{64}$, we combine two such numbers: we shift the first one by one bit to the left and add the lowest bit of the second one.

In [5]:
let runTests (noTests: int) : unit =
    for _ in 1 .. noTests do
        let n = 2I * bigint (rnd.NextInt64()) + bigint (rnd.NextInt64() % 2L)
        let r = isqrt n
        if not (r * r <= n && pown (r + 1I) 2 > n) then
            failwithf "Error: %A != isqrt(%A)" r n

In [6]:
#!time
runTests 1_000_000

Wall time: 2796.5363ms

The function `sqrt2` takes one argument:
- `k` is a natural number.

It returns a string containing $\sqrt{2}$ up to `k` decimal places.

**Implementation:** We have 
$$\sqrt{2 \cdot 10^{2 \cdot k}} = \sqrt{2} \cdot 10^k.$$  
Hence the digits of $\texttt{isqrt}(2 \cdot 10^{2 \cdot k})$ are the first $k+1$ digits of $\sqrt{2}$.  We only have to insert a decimal point after the first digit.

In [7]:
let sqrt2 (k: int) : string =
    let n = 2I * pown 10I (2 * k)
    let s = string (isqrt n)
    s.Substring(0, 1) + "." + s.Substring 1

Let us compute the first $10,000$ digits of $\sqrt{2\,}$.

The depth of the recursion is about $\log_4\bigl(2 \cdot 10^{20000}\bigr) \approx 33\,000$.  Every recursive call needs some memory on the *stack*.  The stack of the thread executing the notebook cells is too small for this recursion depth: the computation would crash the kernel with a *stack overflow*.  In Python, we had to increase the recursion limit for the same reason.  In F#, we instead run the computation on a new *thread* that has a bigger stack.

The function `runWithBigStack` takes two arguments:
- `stackSize` is the size of the stack in bytes,
- `f` is a function that takes no arguments, i.e. its argument is the value `()` of type `unit`.

It computes `f ()` on a new thread with a stack of the given size and returns the result.

**Implementation:** `System.Threading.Thread(g, stackSize)` creates a thread that executes the function `g`.  The thread is started with `Start()` and `Join()` waits until it has finished.  A thread can not return a value directly.  Therefore, `g` stores the value of `f ()` in the *reference cell* `result`.  A reference cell is a mutable container: `ref x` creates a cell containing `x`, and `result.Value` reads or writes its content.  Initially, the cell contains `Unchecked.defaultof<'a>`, i.e. the default value of the type `'a`.

In [8]:
let runWithBigStack (stackSize: int) (f: unit -> 'a) : 'a =
    let result = ref Unchecked.defaultof<'a>
    let thread = System.Threading.Thread((fun () -> result.Value <- f ()), stackSize)
    thread.Start()
    thread.Join()
    result.Value

We use a stack of $256$ megabytes.

In [9]:
#!time
printfn "%s" (runWithBigStack (256 * 1024 * 1024) (fun () -> sqrt2 10_000))

1.41421356237309504880168872420969807856967187537694807317667973799073247846210703885038753432764157273501384623091229702492483605585073721264412149709993583141322266592750559275579995050115278206057147010955997160597027453459686201472851741864088919860955232923048430871432145083976260362799525140798968725339654633180882964062061525835239505474575028775996172983557522033753185701135437460340849884716038689997069900481503054402779031645424782306849293691862158057846311159666871301301561856898723723528850926486124949771542183342042856860601468247207714358548741556570696776537202264854470158588016207584749226572260020855844665214583988939443709265918003113882464681570826301005948587040031864803421948972782906410450726368813137398552561173220402450912277002269411275736272804957381089675040183698683684507257993647290607629969413804756548237289971803268024744206292691248590521810044598421505911202494413417285314781058036033710773091828693147101711116839165817268894197587165821521282295184884

Wall time: 3385.7498ms

You can compare this with the results shown on https://catonmat.net/tools/generate-sqrt2-digits.